# Optimization, rules, and hybrid submissions

Archived from the former Python scripts/package so the Hermes experiments are kept as notebooks.


## Optimised Rules and Ensemble Logic

Selective overrides, tuned decision rules, and optimized classical combination code.

_Archived from `src/icd10_coding/ensemble/optimized.py`._


In [ ]:
"""Step 4: optimized weighted ensemble from matching, retrieval, and SVM."""

import time

import pandas as pd
from rapidfuzz import fuzz, process
from scipy.special import expit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR, ensure_output_dirs


MIN_EXAMPLES = 2
C_VALUE = 2.0
FUZZY_WEIGHT = 1.1
RETRIEVAL_WEIGHT = 1.0
CLASSIFIER_WEIGHT = 1.5


def filter_by_min(df: pd.DataFrame, min_count: int) -> pd.DataFrame:
    counts = df["Code"].value_counts()
    return df[df["Code"].isin(counts[counts >= min_count].index)].copy()


def build_features() -> FeatureUnion:
    return FeatureUnion(
        [
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ]
    )


def train_svm_stack(train: pd.DataFrame):
    train_diag = filter_by_min(train[train.code_type == "diagnosis"], MIN_EXAMPLES)
    train_proc = filter_by_min(train[train.code_type == "procedure"], MIN_EXAMPLES)

    vec_diag = build_features()
    x_diag = vec_diag.fit_transform(train_diag["Literal_norm"])
    clf_diag = LinearSVC(C=C_VALUE, class_weight="balanced", max_iter=3000, random_state=42)
    clf_diag.fit(x_diag, train_diag["Code"].values)

    vec_proc = build_features()
    x_proc = vec_proc.fit_transform(train_proc["Literal_norm"])
    clf_proc = LinearSVC(C=C_VALUE, class_weight="balanced", max_iter=3000, random_state=42)
    clf_proc.fit(x_proc, train_proc["Code"].values)

    vec_router = build_features()
    x_router = vec_router.fit_transform(train["Literal_norm"])
    clf_router = LinearSVC(C=1.0, class_weight="balanced", max_iter=3000, random_state=42)
    clf_router.fit(x_router, train["code_type"].values)

    return vec_diag, clf_diag, vec_proc, clf_proc, vec_router, clf_router


def add_svm_predictions(test: pd.DataFrame, stack) -> pd.DataFrame:
    vec_diag, clf_diag, vec_proc, clf_proc, vec_router, clf_router = stack
    test["router_pred"] = clf_router.predict(vec_router.transform(test["Literal_norm"]))
    test["clf_code"] = ""
    test["clf_decision"] = 0.0

    diag_idx = test[test["router_pred"] == "diagnosis"].index
    if len(diag_idx):
        decisions = clf_diag.decision_function(vec_diag.transform(test.loc[diag_idx, "Literal_norm"]))
        test.loc[diag_idx, "clf_code"] = clf_diag.classes_[decisions.argmax(axis=1)]
        test.loc[diag_idx, "clf_decision"] = decisions.max(axis=1)

    proc_idx = test[test["router_pred"] == "procedure"].index
    if len(proc_idx):
        decisions = clf_proc.decision_function(vec_proc.transform(test.loc[proc_idx, "Literal_norm"]))
        test.loc[proc_idx, "clf_code"] = clf_proc.classes_[decisions.argmax(axis=1)]
        test.loc[proc_idx, "clf_decision"] = decisions.max(axis=1)

    test["clf_confidence"] = expit(test["clf_decision"])
    return test


def rebuild_fuzzy_scores(test: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    exact_lookup = {}
    for lit_norm, group in train.groupby("Literal_norm"):
        exact_lookup[lit_norm] = group["Code"].value_counts().index[0]

    test["exact_code"] = test["Literal_norm"].map(exact_lookup)
    test["fuzzy_code"] = test["exact_code"].copy()
    test["fuzzy_score"] = test["exact_code"].apply(lambda x: 100.0 if pd.notna(x) else 0.0)

    lookup_keys = list(exact_lookup.keys())
    for idx, row in test[test["exact_code"].isna()].iterrows():
        result = process.extractOne(row["Literal_norm"], lookup_keys, scorer=fuzz.ratio, score_cutoff=0)
        if result is not None:
            test.loc[idx, "fuzzy_code"] = exact_lookup[result[0]]
            test.loc[idx, "fuzzy_score"] = result[1]
    return test


def predict_one(row):
    if pd.notna(row["exact_code"]) and row["exact_code"] != "":
        return row["exact_code"], "exact", 1.0

    candidates = []
    if pd.notna(row["fuzzy_code"]) and row["fuzzy_score"] > 0:
        candidates.append((row["fuzzy_score"] / 100.0 * FUZZY_WEIGHT, row["fuzzy_code"], "fuzzy"))
    if pd.notna(row["retrieval_code"]):
        candidates.append((row["retrieval_score"] * RETRIEVAL_WEIGHT, row["retrieval_code"], "retrieval"))
    if pd.notna(row["clf_code"]) and row["clf_code"] != "":
        candidates.append((row["clf_confidence"] * CLASSIFIER_WEIGHT, row["clf_code"], "classifier"))
    if not candidates:
        return "", "none", 0.0
    score, code, method = max(candidates, key=lambda x: x[0])
    return code, method, score


def run() -> None:
    ensure_output_dirs()
    start = time.time()

    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    test = pd.read_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv")
    reference = pd.read_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv")
    step2 = pd.read_csv(PREDICTIONS_DIR / "step2_predictions.csv")

    train["Literal_norm"] = train["Literal_norm"].fillna("")
    test["Literal_norm"] = test["Literal_norm"].fillna("")
    reference["Description_norm"] = reference["Description_norm"].fillna("")

    print("Training optimized SVM stack...")
    stack = train_svm_stack(train)
    test = add_svm_predictions(test, stack)

    test = test.merge(
        step2[["id", "step1_code", "step1_method", "retrieval_code", "retrieval_score"]],
        on="id",
        how="left",
    )
    test = rebuild_fuzzy_scores(test, train)

    results = test.apply(predict_one, axis=1)
    test["final_code"] = [r[0] for r in results]
    test["final_method"] = [r[1] for r in results]
    test["final_confidence"] = [r[2] for r in results]

    tr_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
    tr_split = tr_split.copy()
    val_split = val_split.copy()
    tr_split["Literal_norm"] = tr_split["Literal_norm"].fillna("")
    val_split["Literal_norm"] = val_split["Literal_norm"].fillna("")

    print("Computing validation comparison...")
    v_stack = train_svm_stack(tr_split)
    val_split = add_svm_predictions(val_split, v_stack)
    val_split = rebuild_fuzzy_scores(val_split, tr_split)

    rcv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
    rwv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    rcr = rcv.fit_transform(reference["Description_norm"])
    rwr = rwv.fit_transform(reference["Description_norm"])
    vc = rcv.transform(val_split["Literal_norm"])
    vw = rwv.transform(val_split["Literal_norm"])
    ret_codes, ret_scores = [], []
    for i in range(0, len(val_split), 100):
        end = min(i + 100, len(val_split))
        comb = (cosine_similarity(vc[i:end], rcr) + cosine_similarity(vw[i:end], rwr)) / 2
        best = comb.argmax(axis=1)
        for j in range(end - i):
            bi = best[j]
            ret_codes.append(reference.iloc[bi]["Code"])
            ret_scores.append(float(comb[j, bi]))
        del comb
    val_split["retrieval_code"] = ret_codes
    val_split["retrieval_score"] = ret_scores

    v_results = val_split.apply(predict_one, axis=1)
    val_split["final_code"] = [r[0] for r in v_results]
    val_split["final_correct"] = val_split["final_code"] == val_split["Code"]
    print(f"Optimized ensemble validation accuracy: {val_split.final_correct.mean() * 100:.2f}%")

    detail_cols = [
        "id", "Literal", "Literal_norm", "router_pred",
        "exact_code", "fuzzy_code", "fuzzy_score",
        "retrieval_code", "retrieval_score",
        "clf_code", "clf_confidence",
        "final_code", "final_method", "final_confidence",
    ]
    test[detail_cols].to_csv(PREDICTIONS_DIR / "final_predictions_optimized.csv", index=False)

    submission = test[["id", "final_code"]].copy()
    submission.columns = ["id", "Code"]
    submission.to_csv(SUBMISSIONS_DIR / "kaggle_submission_optimized.csv", index=False)
    print(f"Saved optimized outputs in {time.time() - start:.1f}s")


if __name__ == "__main__":
    run()


## Optimised Ensemble Runner

Former command-line wrapper for the optimized classical system.

_Archived from `scripts/step4_optimized.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.ensemble.optimized import run


if __name__ == "__main__":
    run()


## Hybrid Submission Builder

Code for building the hybrid final submission from multiple prediction sources.

_Archived from `scripts/build_hybrid_chapter_submission.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401

import pandas as pd


OLD_SUBMISSION = PROJECT_ROOT / "comparisson" / "final_submission.csv"
MODEL_DETAIL = PROJECT_ROOT / "output" / "predictions" / "final_chapter_predictions.csv"
OUTPUT_DIR = PROJECT_ROOT / "output" / "submissions"


def build_hybrid(threshold: float) -> None:
    old = pd.read_csv(OLD_SUBMISSION).sort_values("id").reset_index(drop=True)
    ours = pd.read_csv(MODEL_DETAIL).sort_values("id").reset_index(drop=True)

    hybrid = old.copy()
    use_ours = ours["chapter_confidence"] >= threshold
    before = hybrid["y_category"].astype(str)

    # Keep the known-scored submission, but trust our model when its margin is high.
    hybrid.loc[use_ours, "y_category"] = ours.loc[use_ours, "y_category"].values

    changed = before != hybrid["y_category"].astype(str)
    output_path = OUTPUT_DIR / f"kaggle_submission_hybrid_conf_{str(threshold).replace('.', 'p')}.csv"
    hybrid.to_csv(output_path, index=False)

    print(f"{output_path.name}: override={use_ours.sum()}, changed={changed.sum()}")


def main() -> None:
    for threshold in [0.0, 0.278, 0.5, 0.75, 1.0]:
        build_hybrid(threshold)


if __name__ == "__main__":
    main()
